In [ ]:
import os

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from retentionengine import RetentionEngine, RetentionEngineConfig

In [ ]:
# Load the Qwen3-8B base model and tokenizer
basemodel_name = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(basemodel_name)
basemodel = AutoModelForCausalLM.from_pretrained(basemodel_name, torch_dtype=torch.bfloat16)

In [ ]:
# Trimming config: remove a contiguous block of deep layers just before the final layer.
# Qwen3-8B has 36 layers; removing 4 drops layers 31-34 and keeps 0-30 and 35.
# Set `removal_start` to choose another position.
config = RetentionEngineConfig(layers_to_remove=4)

In [ ]:
# Trim the base model (auto_resize=False by default, so only the requested layers are removed)
model = RetentionEngine(basemodel, config)
model.to("cuda")
model.get_model_size_info()

In [ ]:
# Sanity check: the trimmed model still generates.
# Quality is restored afterwards with knowledge distillation (restoration_training.ipynb).
messages = [{"role": "user", "content": "Explain what a transformer layer does in one sentence."}]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, enable_thinking=False, return_tensors="pt", return_dict=True
).to("cuda")
outputs = model.generate(**inputs, max_new_tokens=64, do_sample=False)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
# Save the trimmed model in standard HF format (also loadable with AutoModelForCausalLM.from_pretrained)
save_dir = "pretrained/Qwen3-8B-trimmed"
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)